# 12 — Multimodal Grounding & Evaluation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/12_multimodal_grounding_and_evaluation.ipynb)


## Learning Objectives

By the end of this notebook you will be able to:

- Define **multimodal grounding** and explain why it matters for trustworthy systems
- Implement **evidence attribution** that links generated text to source images or documents
- Build evaluation pipelines for multimodal systems using **automated metrics**
- Detect and measure **hallucination** in vision-language model outputs
- Design a **multimodal evaluation rubric** combining text quality, visual accuracy, and grounding


## What Is Multimodal Grounding?

Grounding means **anchoring generated text to evidence in the input**. In a text-only RAG system, grounding means the answer cites specific retrieved passages. In a multimodal system, grounding extends to visual evidence:

- A VLM describes an image → is the description **faithful** to what's actually in the image?
- A multimodal RAG returns an answer → can we trace it back to a **specific page region** or chart?
- An agent uses vision → did it correctly interpret the visual input before acting?

### Why Grounding Is Harder for Multimodal

Text-to-text grounding is relatively straightforward: check if the answer's claims appear in the source text. Vision grounding is much harder because:

1. **No direct string matching** — you can't ctrl+F an image for "revenue growth"
2. **Interpretation is subjective** — "the chart shows growth" vs "the chart shows a slight upward trend"
3. **Spatial reasoning** — the model must correctly identify which part of the image supports its claim
4. **Compositionality** — combining information from text AND image regions in a single answer

This notebook builds tools to measure and improve multimodal grounding.


In [ ]:
# ── Setup ──────────────────────────────────────────────────────
!pip install -q transformers torch pillow bitsandbytes accelerate \
    numpy rouge-score nltk


In [ ]:
import json, numpy as np, textwrap, re
from pathlib import Path
from PIL import Image, ImageDraw

# ── Synthetic Evaluation Data ─────────────────────────────────
# Each sample: image description (ground truth), model output, and grounding labels
EVAL_SAMPLES = [
    {
        "image_desc": "A bar chart showing Q1=.1B, Q2=.4B, Q3=.8B, Q4=.1B revenue",
        "model_output": "The chart shows steady revenue growth from .1B in Q1 to .1B in Q4, a 48% increase.",
        "grounded_claims": ["Q1=.1B", "Q4=.1B", "steady growth"],
        "hallucinated_claims": [],
        "label": "grounded"
    },
    {
        "image_desc": "A pie chart showing: Cloud 45%, On-prem 30%, Hybrid 25%",
        "model_output": "Cloud dominates at 55%, followed by on-premises at 25% and hybrid at 20%.",
        "grounded_claims": ["Cloud dominates"],
        "hallucinated_claims": ["55%", "25% on-prem", "20% hybrid"],
        "label": "hallucinated"
    },
    {
        "image_desc": "A screenshot of a code editor showing Python with a function called process_data",
        "model_output": "The code shows a Python function called process_data that handles CSV files and returns a DataFrame.",
        "grounded_claims": ["Python function", "process_data"],
        "hallucinated_claims": ["CSV files", "returns a DataFrame"],
        "label": "partial"
    },
    {
        "image_desc": "A photograph of a cat sitting on a windowsill with a city skyline at sunset",
        "model_output": "A tabby cat sits on a windowsill overlooking a city at sunset. The buildings in the background suggest a major metropolitan area.",
        "grounded_claims": ["cat on windowsill", "city", "sunset"],
        "hallucinated_claims": ["tabby", "major metropolitan area"],
        "label": "partial"
    },
]

print(f"Loaded {len(EVAL_SAMPLES)} evaluation samples")
for s in EVAL_SAMPLES:
    print(f"  [{s['label']:>12}] {len(s['grounded_claims'])} grounded, {len(s['hallucinated_claims'])} hallucinated")


## Claim-Level Grounding Evaluation

The most rigorous approach to grounding evaluation works at the **claim level**:

1. **Decompose** the model output into individual claims
2. **Verify** each claim against the source (image description or OCR text)
3. **Classify** each claim as grounded, hallucinated, or unverifiable
4. **Aggregate** into a grounding score

This is analogous to how DeepEval's faithfulness metric works for text RAG, extended to the multimodal setting.


In [ ]:
def extract_claims(text: str) -> list[str]:
    """Split model output into individual verifiable claims.
    Simple heuristic: split on commas, periods, and 'and'.
    In production, use an LLM for claim decomposition."""
    # Split on sentence boundaries
    sentences = re.split(r'[.!]\s*', text)
    claims = []
    for sent in sentences:
        sent = sent.strip()
        if len(sent) > 10:  # Skip tiny fragments
            # Further split on commas for compound claims
            parts = [p.strip() for p in sent.split(',') if len(p.strip()) > 10]
            claims.extend(parts if parts else [sent])
    return claims

def verify_claim(claim: str, reference: str) -> dict:
    """Check if a claim is supported by the reference text.
    Returns a verdict with confidence.
    Simple version: keyword overlap. Production: use an LLM judge."""
    claim_words = set(claim.lower().split())
    ref_words = set(reference.lower().split())
    overlap = len(claim_words & ref_words) / max(len(claim_words), 1)
    
    if overlap > 0.5:
        verdict = 'grounded'
    elif overlap > 0.2:
        verdict = 'partial'
    else:
        verdict = 'hallucinated'
    
    return {'claim': claim, 'verdict': verdict, 'overlap': round(overlap, 3)}

# Test claim extraction and verification
sample = EVAL_SAMPLES[0]
claims = extract_claims(sample['model_output'])
print(f"Claims from: '{sample['model_output'][:60]}...'")
for c in claims:
    v = verify_claim(c, sample['image_desc'])
    print(f"  [{v['verdict']:>12}] (overlap={v['overlap']:.2f}) {c}")


## Grounding Score: From Claims to Numbers

With claim-level verdicts, we can compute aggregate scores:

- **Grounding Rate** = grounded claims / total claims
- **Hallucination Rate** = hallucinated claims / total claims
- **Precision** = correctly attributed claims / all claims made
- **Recall** = correctly attributed claims / all possible true claims

A good multimodal system should have high grounding rate (>0.8) and low hallucination rate (<0.1).


In [ ]:
def compute_grounding_score(model_output: str, reference: str) -> dict:
    """Full grounding evaluation pipeline."""
    claims = extract_claims(model_output)
    if not claims:
        return {'grounding_rate': 0, 'hallucination_rate': 0, 'n_claims': 0}
    
    verdicts = [verify_claim(c, reference) for c in claims]
    
    n_grounded = sum(1 for v in verdicts if v['verdict'] == 'grounded')
    n_hallucinated = sum(1 for v in verdicts if v['verdict'] == 'hallucinated')
    n_partial = sum(1 for v in verdicts if v['verdict'] == 'partial')
    n_total = len(verdicts)
    
    return {
        'grounding_rate': round(n_grounded / n_total, 3),
        'hallucination_rate': round(n_hallucinated / n_total, 3),
        'partial_rate': round(n_partial / n_total, 3),
        'n_claims': n_total,
        'verdicts': verdicts,
    }

# Evaluate all samples
print(f"{'Sample':<10} {'Ground%':>8} {'Halluc%':>8} {'Claims':>7}")
print('-' * 35)
for i, sample in enumerate(EVAL_SAMPLES):
    score = compute_grounding_score(sample['model_output'], sample['image_desc'])
    print(f"Sample {i+1:<3}  {score['grounding_rate']:>7.1%}  {score['hallucination_rate']:>7.1%}  {score['n_claims']:>6}")


## Hallucination Detection Patterns

Vision-language models hallucinate in predictable ways. Knowing the patterns helps you build better detectors:

### Common Hallucination Types

| Type | Example | Detection Strategy |
|------|---------|-------------------|
| **Numeric fabrication** | Chart shows 45% but model says 55% | Extract numbers, compare against source |
| **Entity invention** | Model names a person not in the image | Named entity extraction + verification |
| **Attribute hallucination** | "a red car" when the car is blue | Color/attribute extraction from image |
| **Spatial confabulation** | "to the left of" when it's actually right | Spatial relation verification |
| **Over-interpretation** | Inferring "major metropolitan area" from a skyline | Claim specificity analysis |
| **Knowledge injection** | Adding facts not in the image | Source-only verification |

The **most dangerous** hallucinations are numeric — they look plausible but are factually wrong. Always verify numbers against source data.


In [ ]:
def detect_numeric_hallucination(model_output: str, reference: str) -> list[dict]:
    """Detect hallucinated numbers by comparing model output to reference."""
    # Extract all numbers from both texts
    model_numbers = set(re.findall(r'\True\d+\.?\d*[%B]?', model_output))
    ref_numbers = set(re.findall(r'\True\d+\.?\d*[%B]?', reference))
    
    issues = []
    for num in model_numbers:
        if num not in ref_numbers:
            issues.append({
                'type': 'numeric_hallucination',
                'value': num,
                'in_model': True,
                'in_reference': False,
                'severity': 'high'
            })
    return issues

# Test on our samples
for i, sample in enumerate(EVAL_SAMPLES):
    issues = detect_numeric_hallucination(sample['model_output'], sample['image_desc'])
    if issues:
        print(f"\nSample {i+1} — numeric hallucinations:")
        for iss in issues:
            print(f"  ⚠️  {iss['value']} appears in output but not in reference")
    else:
        print(f"Sample {i+1} — no numeric hallucinations ✓")


## Building a Multimodal Evaluation Rubric

A comprehensive multimodal evaluation combines multiple dimensions:

### The GRAFT Framework (Grounding, Relevance, Accuracy, Fluency, Thoroughness)

| Dimension | What It Measures | Weight |
|-----------|-----------------|--------|
| **Grounding** | Is every claim traceable to the input? | 30% |
| **Relevance** | Does the output address the query? | 25% |
| **Accuracy** | Are facts (especially numbers) correct? | 25% |
| **Fluency** | Is the output well-written and coherent? | 10% |
| **Thoroughness** | Does it cover all important information? | 10% |

Each dimension gets a 1-5 score. The weighted sum gives an overall quality score.


In [ ]:
def evaluate_multimodal_output(
    model_output: str,
    reference: str,
    query: str = ""
) -> dict:
    """Multi-dimensional evaluation of multimodal model output."""
    # Grounding (claim-level)
    grounding = compute_grounding_score(model_output, reference)
    grounding_score = grounding['grounding_rate'] * 5
    
    # Relevance (query-output overlap)
    if query:
        q_words = set(query.lower().split())
        o_words = set(model_output.lower().split())
        relevance_score = min(len(q_words & o_words) / max(len(q_words), 1) * 5, 5.0)
    else:
        relevance_score = 3.0  # Default when no query
    
    # Accuracy (inverse hallucination rate)
    accuracy_score = (1 - grounding['hallucination_rate']) * 5
    
    # Fluency (sentence count, avg length — crude proxy)
    sentences = [s for s in model_output.split('.') if len(s.strip()) > 5]
    avg_len = np.mean([len(s.split()) for s in sentences]) if sentences else 0
    fluency_score = min(avg_len / 4, 5.0)  # ~20 words per sentence = 5.0
    
    # Thoroughness (output length relative to reference)
    coverage = len(model_output.split()) / max(len(reference.split()), 1)
    thoroughness_score = min(coverage * 3, 5.0)
    
    # Weighted aggregate
    weights = {'grounding': 0.30, 'relevance': 0.25, 'accuracy': 0.25,
               'fluency': 0.10, 'thoroughness': 0.10}
    scores = {
        'grounding': round(grounding_score, 2),
        'relevance': round(relevance_score, 2),
        'accuracy': round(accuracy_score, 2),
        'fluency': round(fluency_score, 2),
        'thoroughness': round(thoroughness_score, 2),
    }
    overall = sum(scores[k] * weights[k] for k in weights)
    scores['overall'] = round(overall, 2)
    
    return scores

# Evaluate all samples
print(f"{'':>10} {'Ground':>7} {'Relev':>7} {'Accur':>7} {'Fluen':>7} {'Thorou':>7} {'OVERALL':>8}")
print('-' * 58)
for i, sample in enumerate(EVAL_SAMPLES):
    scores = evaluate_multimodal_output(
        sample['model_output'], sample['image_desc'],
        query="Describe what the image shows"
    )
    print(f"Sample {i+1}  {scores['grounding']:>7.2f} {scores['relevance']:>7.2f} "
          f"{scores['accuracy']:>7.2f} {scores['fluency']:>7.2f} {scores['thoroughness']:>7.2f} "
          f"{scores['overall']:>7.2f}")


## LLM-as-Judge for Multimodal Grounding

The gold standard for grounding evaluation uses an **LLM judge** — a separate model that evaluates whether the output is faithful to the visual input.

The judge prompt follows this pattern:

`
Given this image description: {reference}
And this model output: {model_output}

For each claim in the output, classify it as:
- GROUNDED: directly supported by the image
- HALLUCINATED: contradicted by or absent from the image
- UNVERIFIABLE: cannot be confirmed from the image alone
`

This approach is more nuanced than keyword matching but requires careful prompt engineering to get consistent verdicts. See **evals/07** for more on LLM-as-judge design.


In [ ]:
def build_judge_prompt(model_output: str, reference: str) -> str:
    """Build a prompt for LLM-as-judge grounding evaluation."""
    return f"""You are evaluating whether a model's description is faithful to an image.

IMAGE DESCRIPTION (ground truth):
{reference}

MODEL OUTPUT (to evaluate):
{model_output}

For each claim in the model output, classify it as:
- GROUNDED: directly supported by the image description
- HALLUCINATED: contradicted by or not present in the image description
- UNVERIFIABLE: plausible but cannot be confirmed

Respond in JSON format:
{{
  "claims": [
    {{"text": "...", "verdict": "GROUNDED|HALLUCINATED|UNVERIFIABLE", "reason": "..."}}
  ],
  "overall_grounding_score": 0.0-1.0,
  "summary": "..."
}}"""

# Show example prompt
sample = EVAL_SAMPLES[1]  # The one with hallucinated numbers
prompt = build_judge_prompt(sample['model_output'], sample['image_desc'])
print(prompt[:500])
print("...")


## Evaluation at Scale: Batch Processing

For a real multimodal system, you need to evaluate hundreds or thousands of outputs. Here's a batch evaluation pipeline that produces a summary report.


In [ ]:
def batch_evaluate(samples: list[dict]) -> dict:
    """Run full evaluation across a batch of samples."""
    results = []
    for sample in samples:
        scores = evaluate_multimodal_output(
            sample['model_output'],
            sample['image_desc'],
            query="Describe what the image shows"
        )
        grounding = compute_grounding_score(sample['model_output'], sample['image_desc'])
        numeric_issues = detect_numeric_hallucination(sample['model_output'], sample['image_desc'])
        
        results.append({
            'label': sample['label'],
            'scores': scores,
            'grounding_rate': grounding['grounding_rate'],
            'hallucination_rate': grounding['hallucination_rate'],
            'n_numeric_issues': len(numeric_issues),
        })
    
    # Aggregate
    avg_overall = np.mean([r['scores']['overall'] for r in results])
    avg_grounding = np.mean([r['grounding_rate'] for r in results])
    avg_hallucination = np.mean([r['hallucination_rate'] for r in results])
    total_numeric = sum(r['n_numeric_issues'] for r in results)
    
    report = {
        'n_samples': len(samples),
        'avg_overall_score': round(avg_overall, 3),
        'avg_grounding_rate': round(avg_grounding, 3),
        'avg_hallucination_rate': round(avg_hallucination, 3),
        'total_numeric_hallucinations': total_numeric,
        'results': results,
    }
    return report

report = batch_evaluate(EVAL_SAMPLES)
print("\n📊 Evaluation Report")
print(f"  Samples: {report['n_samples']}")
print(f"  Avg Overall Score: {report['avg_overall_score']:.3f} / 5.0")
print(f"  Avg Grounding Rate: {report['avg_grounding_rate']:.1%}")
print(f"  Avg Hallucination Rate: {report['avg_hallucination_rate']:.1%}")
print(f"  Numeric Hallucinations: {report['total_numeric_hallucinations']}")


## Connecting Grounding to the Eval Pipeline

This notebook's evaluation tools connect to the broader evaluation framework in the course:

- **evals/01-04**: General evaluation foundations — metrics, datasets, error analysis
- **evals/09**: Retrieval metrics (Recall@k, MRR) — used for the retrieval branch
- **evals/13-15**: LLM-as-judge — used for claim-level grounding verification
- **multimodal/07**: Structured output extraction — the grounding verification target
- **multimodal/11**: Hybrid retrieval — the retrieval component to evaluate

In production, multimodal evaluation is **continuous** — run these checks on every model update, every prompt change, and every new document type.


## Exercises

### Exercise 1: LLM Judge Integration
Load Qwen3-8B and use it as an actual LLM judge. Compare its grounding verdicts against the ground truth labels in EVAL_SAMPLES. How accurate is the judge?

### Exercise 2: Visual Attribute Verification
Extend detect_numeric_hallucination to also detect **color hallucinations** and **spatial hallucinations**. Create test cases where the model describes objects with wrong colors or positions.

### Exercise 3: End-to-End Pipeline Evaluation
Build a complete evaluation pipeline: take a query, run hybrid retrieval (from notebook 11), generate an answer with a VLM, then evaluate the answer with the GRAFT rubric from this notebook.


## Key Takeaways

- **Multimodal grounding** = every generated claim must be traceable to visual or textual evidence
- **Claim-level evaluation** is more informative than output-level — it pinpoints exactly what's wrong
- **Numeric hallucinations** are the most dangerous and most detectable type
- The **GRAFT framework** (Grounding, Relevance, Accuracy, Fluency, Thoroughness) provides structured evaluation
- **LLM-as-judge** is the gold standard for grounding verification but needs careful prompt design
- Evaluation should be **continuous** and **automated** in production multimodal systems


## References

- Li, J., et al. (2023). *Evaluating Object Hallucination in Large Vision-Language Models*. EMNLP.
- Liu, F., et al. (2023). *Mitigating Hallucination in Large Multi-Modal Models via Robust Instruction Tuning*.
- Min, S., et al. (2023). *FActScore: Fine-grained Atomic Evaluation of Factual Precision*. EMNLP.
- Zheng, L., et al. (2024). *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena*. NeurIPS.


---

[← 11 Hybrid Text + Vision Retrieval](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/11_hybrid_text_plus_vision_retrieval.ipynb) | [13 Speech Recognition & Transcription →](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/multimodal/13_speech_recognition_and_transcription.ipynb)
